In [4]:
# data_preprocessing.py

import pandas as pd
import os

# Define the path to your data directory
# Adjust this path if your structure is different or you're using the full dataset
DATA_DIR = 'data/MINDsmall_train' # Or 'data/MINDlarge_train', etc.
NEWS_FILE = 'news.tsv'
BEHAVIORS_FILE = 'behaviors.tsv'

def load_mind_data(data_dir):
    """
    Loads the news and behaviors data from the specified directory.

    Args:
        data_dir (str): The path to the directory containing MIND data files.

    Returns:
        tuple: A tuple containing two pandas DataFrames: (news_df, behaviors_df)
               Returns (None, None) if files are not found.
    """
    news_path = os.path.join(data_dir, NEWS_FILE)
    behaviors_path = os.path.join(data_dir, BEHAVIORS_FILE)

    if not os.path.exists(news_path):
        print(f"Error: News file not found at {news_path}")
        return None, None
    if not os.path.exists(behaviors_path):
        print(f"Error: Behaviors file not found at {behaviors_path}")
        return None, None

    # Define column names for news.tsv based on MIND documentation
    news_columns = [
        'news_id', 'category', 'subcategory', 'title', 'abstract',
        'url', 'title_entities', 'abstract_entities'
    ]
    # Define column names for behaviors.tsv
    behaviors_columns = [
        'impression_id', 'user_id', 'time', 'history', 'impressions'
    ]

    print(f"Loading news data from: {news_path}")
    # Load news.tsv without header, specifying custom column names
    news_df = pd.read_csv(news_path, sep='\t', header=None, names=news_columns)
    print(f"Loaded {len(news_df)} news articles.")

    print(f"Loading behaviors data from: {behaviors_path}")
    # Load behaviors.tsv without header, specifying custom column names
    behaviors_df = pd.read_csv(behaviors_path, sep='\t', header=None, names=behaviors_columns)
    print(f"Loaded {len(behaviors_df)} behavior logs (impressions).")

    return news_df, behaviors_df

# --- Example Usage ---
if __name__ == "__main__":
    print("Starting data loading process...")
    news_df, behaviors_df = load_mind_data(DATA_DIR)

    if news_df is not None and behaviors_df is not None:
        print("\n--- News DataFrame Head ---")
        print(news_df.head())
        print("\n--- News DataFrame Info ---")
        news_df.info()

        print("\n--- Behaviors DataFrame Head ---")
        print(behaviors_df.head())
        print("\n--- Behaviors DataFrame Info ---")
        behaviors_df.info()

        # Example: Look at the 'impressions' column format for the first user
        print("\n--- Example Impression Log ---")
        if not behaviors_df.empty:
            print(behaviors_df.iloc[0]['impressions'])
            print("\nFormat: news_id-click (1 if clicked, 0 if not clicked)")
    else:
        print("Data loading failed. Please check file paths and ensure files exist.")

Starting data loading process...
Loading news data from: data/MINDsmall_train/news.tsv
Loaded 51282 news articles.
Loading behaviors data from: data/MINDsmall_train/behaviors.tsv
Loaded 156965 behavior logs (impressions).

--- News DataFrame Head ---
  news_id   category      subcategory  \
0  N55528  lifestyle  lifestyleroyals   
1  N19639     health       weightloss   
2  N61837       news        newsworld   
3  N53526     health           voices   
4  N38324     health          medical   

                                               title  \
0  The Brands Queen Elizabeth, Prince Charles, an...   
1                      50 Worst Habits For Belly Fat   
2  The Cost of Trump's Aid Freeze in the Trenches...   
3  I Was An NBA Wife. Here's How It Affected My M...   
4  How to Get Rid of Skin Tags, According to a De...   

                                            abstract  \
0  Shop the notebooks, jackets, and more that the...   
1  These seemingly harmless habits are holding yo... 

In [7]:
# data_preprocessing.py

import pandas as pd
import os
from tqdm import tqdm # Import tqdm for progress bar

# Define the path to your data directory
DATA_DIR = 'data/MINDsmall_train' # Or 'data/MINDlarge_train', etc.
NEWS_FILE = 'news.tsv'
BEHAVIORS_FILE = 'behaviors.tsv'

# --- load_mind_data function (from previous step) ---
def load_mind_data(data_dir):
    """
    Loads the news and behaviors data from the specified directory.
    (Implementation from previous step - keeping it concise here)
    """
    news_path = os.path.join(data_dir, NEWS_FILE)
    behaviors_path = os.path.join(data_dir, BEHAVIORS_FILE)

    if not os.path.exists(news_path) or not os.path.exists(behaviors_path):
        print("Error: Data files not found.")
        return None, None

    news_columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities']
    behaviors_columns = ['impression_id', 'user_id', 'time', 'history', 'impressions']

    print(f"Loading news data from: {news_path}")
    news_df = pd.read_csv(news_path, sep='\t', header=None, names=news_columns)
    print(f"Loaded {len(news_df)} news articles.")

    print(f"Loading behaviors data from: {behaviors_path}")
    behaviors_df = pd.read_csv(behaviors_path, sep='\t', header=None, names=behaviors_columns)
    print(f"Loaded {len(behaviors_df)} behavior logs (impressions).")

    return news_df, behaviors_df

# --- NEW Function: Parse Impressions ---
def parse_impressions(impressions_str):
    """
    Parses the impression string into a list of (news_id, click_label) tuples.
    Handles potential NaN values.
    """
    if pd.isna(impressions_str):
        return [] # Return empty list for missing impressions
    
    impression_list = []
    parts = impressions_str.strip().split(' ')
    for part in parts:
        if '-' in part:
            try:
                news_id, click_str = part.split('-')
                click_label = int(click_str)
                impression_list.append({'news_id': news_id, 'click': click_label})
            except ValueError:
                # Handle cases where splitting might fail unexpectedly
                print(f"Warning: Could not parse impression part: {part}")
                continue # Skip this malformed part
        else:
             # Handle cases where there might not be a click label (though unusual for MIND)
             print(f"Warning: Impression part without click label found: {part}")
             # Decide how to handle this - maybe assume not clicked? Or skip?
             # impression_list.append({'news_id': part, 'click': 0}) # Example: assume not clicked
             continue # For now, let's skip if format is unexpected
             
    return impression_list

# --- Example Usage (Modified) ---
if __name__ == "__main__":
    print("Starting data loading process...")
    news_df, behaviors_df = load_mind_data(DATA_DIR)

    if news_df is not None and behaviors_df is not None:
        print("\n--- News DataFrame Head ---")
        print(news_df.head(2))

        print("\n--- Behaviors DataFrame Head (Before Parsing) ---")
        print(behaviors_df.head(2))

        # --- Apply the parsing function ---
        print("\nParsing impressions column (this may take a moment)...")
        # Initialize tqdm for pandas apply
        tqdm.pandas(desc="Parsing Impressions") 
        # Use progress_apply for a progress bar
        behaviors_df['parsed_impressions'] = behaviors_df['impressions'].progress_apply(parse_impressions)
        print("Finished parsing impressions.")

        print("\n--- Behaviors DataFrame Head (After Parsing) ---")
        # Show the original and the new parsed column
        print(behaviors_df[['user_id', 'history', 'impressions', 'parsed_impressions']].head())

        print("\n--- Example Parsed Impression ---")
        if not behaviors_df.empty:
             print(f"Original: {behaviors_df.iloc[0]['impressions']}")
             print(f"Parsed:   {behaviors_df.iloc[0]['parsed_impressions']}")

        # Optional: Explode the DataFrame (can consume significant memory!)
        # print("\nExploding DataFrame based on parsed impressions...")
        # behaviors_exploded_df = behaviors_df.explode('parsed_impressions')
        # # Extract news_id and click into separate columns after exploding
        # behaviors_exploded_df['news_id'] = behaviors_exploded_df['parsed_impressions'].apply(lambda x: x['news_id'] if isinstance(x, dict) else None)
        # behaviors_exploded_df['click'] = behaviors_exploded_df['parsed_impressions'].apply(lambda x: x['click'] if isinstance(x, dict) else None)
        # # Drop the intermediate parsed column if needed
        # behaviors_exploded_df = behaviors_exploded_df.drop(columns=['parsed_impressions', 'impressions']) 
        # print("DataFrame exploded.")
        # print(behaviors_exploded_df[['user_id', 'time', 'history', 'news_id', 'click']].head())
        # print(f"Original behaviors rows: {len(behaviors_df)}, Exploded rows: {len(behaviors_exploded_df)}")

    else:
        print("Data loading failed. Please check file paths and ensure files exist.")

Starting data loading process...
Loading news data from: data/MINDsmall_train/news.tsv
Loaded 51282 news articles.
Loading behaviors data from: data/MINDsmall_train/behaviors.tsv
Loaded 156965 behavior logs (impressions).

--- News DataFrame Head ---
  news_id   category      subcategory  \
0  N55528  lifestyle  lifestyleroyals   
1  N19639     health       weightloss   

                                               title  \
0  The Brands Queen Elizabeth, Prince Charles, an...   
1                      50 Worst Habits For Belly Fat   

                                            abstract  \
0  Shop the notebooks, jackets, and more that the...   
1  These seemingly harmless habits are holding yo...   

                                             url  \
0  https://assets.msn.com/labs/mind/AAGH0ET.html   
1  https://assets.msn.com/labs/mind/AAB19MK.html   

                                      title_entities  \
0  [{"Label": "Prince Philip, Duke of Edinburgh",...   
1  [{"Label": "Adi

Parsing Impressions: 100%|██████████| 156965/156965 [00:01<00:00, 129044.64it/s]

Finished parsing impressions.

--- Behaviors DataFrame Head (After Parsing) ---
  user_id                                            history  \
0  U13740  N55189 N42782 N34694 N45794 N18445 N63302 N104...   
1  U91836  N31739 N6072 N63045 N23979 N35656 N43353 N8129...   
2  U73700  N10732 N25792 N7563 N21087 N41087 N5445 N60384...   
3  U34670  N45729 N2203 N871 N53880 N41375 N43142 N33013 ...   
4   U8125                        N10078 N56514 N14904 N33740   

                                         impressions  \
0                                  N55689-1 N35729-0   
1  N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...   
2  N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...   
3                N35729-0 N33632-0 N49685-1 N27581-0   
4  N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...   

                                  parsed_impressions  
0  [{'news_id': 'N55689', 'click': 1}, {'news_id'...  
1  [{'news_id': 'N20678', 'click': 0}, {'news_id'...  
2  [{'news_id': 'N50014', 'click'

In [8]:
# data_preprocessing.py

import pandas as pd
import os
from tqdm import tqdm # Use tqdm for progress bars

# Define paths (as before)
DATA_DIR = 'data/MINDsmall_train'
NEWS_FILE = 'news.tsv'
BEHAVIORS_FILE = 'behaviors.tsv'

# --- load_mind_data function (from previous step) ---
def load_mind_data(data_dir):
    # (Implementation remains the same)
    news_path = os.path.join(data_dir, NEWS_FILE)
    behaviors_path = os.path.join(data_dir, BEHAVIORS_FILE)
    if not os.path.exists(news_path) or not os.path.exists(behaviors_path):
        print("Error: Data files not found.")
        return None, None
    news_columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities']
    behaviors_columns = ['impression_id', 'user_id', 'time', 'history', 'impressions']
    print(f"Loading news data from: {news_path}")
    news_df = pd.read_csv(news_path, sep='\t', header=None, names=news_columns)
    print(f"Loaded {len(news_df)} news articles.")
    print(f"Loading behaviors data from: {behaviors_path}")
    behaviors_df = pd.read_csv(behaviors_path, sep='\t', header=None, names=behaviors_columns)
    print(f"Loaded {len(behaviors_df)} behavior logs (impressions).")
    return news_df, behaviors_df

# --- parse_impressions function (from previous step) ---
def parse_impressions(impressions_str):
    # (Implementation remains the same)
    if pd.isna(impressions_str): return []
    impression_list = []
    parts = impressions_str.strip().split(' ')
    for part in parts:
        if '-' in part:
            try:
                news_id, click_str = part.split('-')
                click_label = int(click_str)
                impression_list.append({'news_id': news_id, 'click': click_label})
            except ValueError: continue
        else: continue
    return impression_list

# --- NEW Function: Parse History ---
def parse_history(history_str):
    """
    Parses the history string into a list of news_ids.
    Handles potential NaN values (users with no history).
    """
    if pd.isna(history_str):
        return [] # Return empty list for users with no history
    else:
        return history_str.strip().split(' ')

# --- Example Usage (Modified) ---
if __name__ == "__main__":
    print("Starting data loading process...")
    news_df, behaviors_df = load_mind_data(DATA_DIR)

    if news_df is not None and behaviors_df is not None:
        # --- Apply impression parsing (as before) ---
        print("\nParsing impressions column...")
        tqdm.pandas(desc="Parsing Impressions")
        behaviors_df['parsed_impressions'] = behaviors_df['impressions'].progress_apply(parse_impressions)
        print("Finished parsing impressions.")

        # --- Apply history parsing ---
        print("\nParsing history column...")
        tqdm.pandas(desc="Parsing History")
        behaviors_df['parsed_history'] = behaviors_df['history'].progress_apply(parse_history)
        print("Finished parsing history.")


        print("\n--- Behaviors DataFrame Head (After Parsing History) ---")
        # Show relevant columns including the new parsed_history
        print(behaviors_df[['user_id', 'history', 'parsed_history', 'impressions', 'parsed_impressions']].head())

        print("\n--- Example Parsed History ---")
        if not behaviors_df.empty:
            example_idx = 0 # Look at the first row
            # Find first row with non-empty history if the first one is empty
            for i in range(len(behaviors_df)):
                if behaviors_df.iloc[i]['history']:
                     example_idx = i
                     break
            print(f"Original History: {behaviors_df.iloc[example_idx]['history']}")
            print(f"Parsed History:   {behaviors_df.iloc[example_idx]['parsed_history']}")
            print(f"Number of items in history: {len(behaviors_df.iloc[example_idx]['parsed_history'])}")

        # Optional: Exploding (as before, commented out)
        # ...

    else:
        print("Data loading failed.")

Starting data loading process...
Loading news data from: data/MINDsmall_train/news.tsv
Loaded 51282 news articles.
Loading behaviors data from: data/MINDsmall_train/behaviors.tsv
Loaded 156965 behavior logs (impressions).

Parsing impressions column...


Parsing Impressions: 100%|██████████| 156965/156965 [00:01<00:00, 128367.95it/s]


Finished parsing impressions.

Parsing history column...


Parsing History: 100%|██████████████| 156965/156965 [00:00<00:00, 457686.67it/s]

Finished parsing history.

--- Behaviors DataFrame Head (After Parsing History) ---
  user_id                                            history  \
0  U13740  N55189 N42782 N34694 N45794 N18445 N63302 N104...   
1  U91836  N31739 N6072 N63045 N23979 N35656 N43353 N8129...   
2  U73700  N10732 N25792 N7563 N21087 N41087 N5445 N60384...   
3  U34670  N45729 N2203 N871 N53880 N41375 N43142 N33013 ...   
4   U8125                        N10078 N56514 N14904 N33740   

                                      parsed_history  \
0  [N55189, N42782, N34694, N45794, N18445, N6330...   
1  [N31739, N6072, N63045, N23979, N35656, N43353...   
2  [N10732, N25792, N7563, N21087, N41087, N5445,...   
3  [N45729, N2203, N871, N53880, N41375, N43142, ...   
4                   [N10078, N56514, N14904, N33740]   

                                         impressions  \
0                                  N55689-1 N35729-0   
1  N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...   
2  N50014-0 N23877-0 N3538

In [9]:
# data_preprocessing.py

import pandas as pd
import os
from tqdm import tqdm

# Define paths (as before)
DATA_DIR = 'data/MINDsmall_train'
NEWS_FILE = 'news.tsv'
BEHAVIORS_FILE = 'behaviors.tsv'

# --- load_mind_data function (as before) ---
def load_mind_data(data_dir):
    # (Implementation remains the same)
    news_path = os.path.join(data_dir, NEWS_FILE)
    behaviors_path = os.path.join(data_dir, BEHAVIORS_FILE)
    if not os.path.exists(news_path) or not os.path.exists(behaviors_path):
        print("Error: Data files not found.")
        return None, None
    news_columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities']
    behaviors_columns = ['impression_id', 'user_id', 'time', 'history', 'impressions']
    print(f"Loading news data from: {news_path}")
    news_df = pd.read_csv(news_path, sep='\t', header=None, names=news_columns)
    print(f"Loaded {len(news_df)} news articles.")
    print(f"Loading behaviors data from: {behaviors_path}")
    behaviors_df = pd.read_csv(behaviors_path, sep='\t', header=None, names=behaviors_columns)
    print(f"Loaded {len(behaviors_df)} behavior logs (impressions).")
    return news_df, behaviors_df

# --- parse_impressions function (as before) ---
def parse_impressions(impressions_str):
    # (Implementation remains the same)
    if pd.isna(impressions_str): return []
    impression_list = []
    parts = impressions_str.strip().split(' ')
    for part in parts:
        if '-' in part:
            try:
                news_id, click_str = part.split('-')
                click_label = int(click_str)
                impression_list.append({'news_id': news_id, 'click': click_label})
            except ValueError: continue
        else: continue
    return impression_list

# --- parse_history function (as before) ---
def parse_history(history_str):
    # (Implementation remains the same)
    if pd.isna(history_str): return []
    else: return history_str.strip().split(' ')

# --- Example Usage (Modified for Exploding and Merging) ---
if __name__ == "__main__":
    print("Starting data loading process...")
    news_df, behaviors_df = load_mind_data(DATA_DIR)

    if news_df is not None and behaviors_df is not None:
        # --- Apply impression parsing ---
        print("\nParsing impressions column...")
        tqdm.pandas(desc="Parsing Impressions")
        behaviors_df['parsed_impressions'] = behaviors_df['impressions'].progress_apply(parse_impressions)

        # --- Apply history parsing ---
        print("\nParsing history column...")
        tqdm.pandas(desc="Parsing History")
        behaviors_df['parsed_history'] = behaviors_df['history'].progress_apply(parse_history)
        print("Finished parsing.")

        # --- Select relevant columns before exploding ---
        # Keep user_id, time, parsed_history, and the impressions to explode
        # impression_id might also be useful depending on the task
        behaviors_subset_df = behaviors_df[['user_id', 'time', 'parsed_history', 'parsed_impressions']].copy()
        print(f"\nSelected columns for exploding. Rows: {len(behaviors_subset_df)}")

        # --- Explode DataFrame based on parsed impressions ---
        # This creates a row for each article in an impression list
        print("\nExploding DataFrame based on parsed impressions (can take time and memory)...")
        behaviors_exploded_df = behaviors_subset_df.explode('parsed_impressions', ignore_index=True)
        print(f"DataFrame exploded. New rows: {len(behaviors_exploded_df)}")

        # --- Extract news_id and click from the exploded dictionaries ---
        print("Extracting news_id and click from exploded column...")
        # Use .loc to avoid SettingWithCopyWarning
        behaviors_exploded_df.loc[:, 'news_id'] = behaviors_exploded_df['parsed_impressions'].apply(lambda x: x['news_id'] if isinstance(x, dict) else None)
        behaviors_exploded_df.loc[:, 'click'] = behaviors_exploded_df['parsed_impressions'].apply(lambda x: x['click'] if isinstance(x, dict) else None)

        # --- Drop the intermediate parsed column and handle potential Nones ---
        behaviors_exploded_df = behaviors_exploded_df.drop(columns=['parsed_impressions'])
        behaviors_exploded_df = behaviors_exploded_df.dropna(subset=['news_id']) # Drop rows where extraction failed
        # Convert click to integer type
        behaviors_exploded_df['click'] = behaviors_exploded_df['click'].astype(int)


        print(f"Columns extracted. Rows after potential dropna: {len(behaviors_exploded_df)}")
        print("\n--- Exploded Behaviors DataFrame Head ---")
        print(behaviors_exploded_df.head())

        # --- Merge with News DataFrame ---
        print("\nMerging exploded behaviors with news data...")
        # Select only needed columns from news_df to avoid large merge
        news_subset_df = news_df[['news_id', 'category', 'subcategory', 'title', 'abstract']].copy()

        # Perform the merge
        merged_df = pd.merge(
            behaviors_exploded_df,
            news_subset_df,
            on='news_id',
            how='left' # Keep all rows from behaviors_exploded_df
        )
        print(f"Merge complete. Final DataFrame rows: {len(merged_df)}")

        # --- Display final merged data ---
        print("\n--- Final Merged DataFrame Head ---")
        print(merged_df.head())

        print("\n--- Final Merged DataFrame Info ---")
        merged_df.info()

        # Example: Check info for a specific user
        # print("\n--- Example: Data for User U13740 ---")
        # print(merged_df[merged_df['user_id'] == 'U13740'])

    else:
        print("Data loading failed.")

Starting data loading process...
Loading news data from: data/MINDsmall_train/news.tsv
Loaded 51282 news articles.
Loading behaviors data from: data/MINDsmall_train/behaviors.tsv
Loaded 156965 behavior logs (impressions).

Parsing impressions column...


Parsing Impressions: 100%|██████████| 156965/156965 [00:01<00:00, 138207.36it/s]



Parsing history column...


Parsing History: 100%|██████████████| 156965/156965 [00:00<00:00, 473660.64it/s]


Finished parsing.

Selected columns for exploding. Rows: 156965

Exploding DataFrame based on parsed impressions (can take time and memory)...
DataFrame exploded. New rows: 5843444
Extracting news_id and click from exploded column...
Columns extracted. Rows after potential dropna: 5843444

--- Exploded Behaviors DataFrame Head ---
  user_id                   time  \
0  U13740  11/11/2019 9:05:58 AM   
1  U13740  11/11/2019 9:05:58 AM   
2  U91836  11/12/2019 6:11:30 PM   
3  U91836  11/12/2019 6:11:30 PM   
4  U91836  11/12/2019 6:11:30 PM   

                                      parsed_history news_id  click  
0  [N55189, N42782, N34694, N45794, N18445, N6330...  N55689      1  
1  [N55189, N42782, N34694, N45794, N18445, N6330...  N35729      0  
2  [N31739, N6072, N63045, N23979, N35656, N43353...  N20678      0  
3  [N31739, N6072, N63045, N23979, N35656, N43353...  N39317      0  
4  [N31739, N6072, N63045, N23979, N35656, N43353...  N58114      0  

Merging exploded behaviors wi

In [10]:
# data_preprocessing.py

import pandas as pd
import os
from tqdm import tqdm

# --- All previous code (loading, parsing, exploding, merging) goes here ---
# --- Make sure news_df and merged_df are available                  ---

# --- NEW: State Representation ---

def create_state_representation(df, news_info_df, k=1):
    """
    Creates a state representation based on the category of the last k items
    in the user's click history.

    Args:
        df (pd.DataFrame): The DataFrame containing 'parsed_history' (list of news IDs).
        news_info_df (pd.DataFrame): DataFrame with 'news_id' and 'category' columns.
        k (int): Number of recent history items to consider for the state.

    Returns:
        pd.DataFrame: Original DataFrame with an added 'state' column.
    """
    print(f"\nCreating state representation using last {k} history item(s)...")

    # 1. Create a mapping for quick category lookup
    print("Creating news_id -> category lookup map...")
    # Fill potential NaN categories with a placeholder before creating map
    news_info_df['category'] = news_info_df['category'].fillna('UNKNOWN') 
    id_to_cat_map = news_info_df.set_index('news_id')['category'].to_dict()
    print("Lookup map created.")

    # 2. Define the state extraction function
    def get_state_from_history(history_list):
        if not history_list: # Handle empty history (cold start)
            return "START" #"<START>" if tuple is required
        else:
            # Get last k news IDs from history
            last_k_ids = history_list[-k:]
            # Look up categories
            state_categories = []
            for news_id in last_k_ids:
                category = id_to_cat_map.get(news_id, "UNKNOWN_CAT") # Use UNKNOWN_CAT if ID not found
                state_categories.append(category)
            
            # Return tuple (hashable for dict keys later)
            return tuple(state_categories) 

    # 3. Apply the function to create the 'state' column
    print("Applying state creation function...")
    tqdm.pandas(desc="Creating State")
    df['state'] = df['parsed_history'].progress_apply(get_state_from_history)
    print("State column created.")

    return df

# --- Example Usage (Continuing from previous step) ---
if __name__ == "__main__":
    # --- (Previous loading, parsing, exploding, merging code) ---
    print("Starting data loading process...")
    news_df, behaviors_df = load_mind_data(DATA_DIR)

    if news_df is not None and behaviors_df is not None:
        print("\nParsing impressions column...")
        tqdm.pandas(desc="Parsing Impressions")
        behaviors_df['parsed_impressions'] = behaviors_df['impressions'].progress_apply(parse_impressions)

        print("\nParsing history column...")
        tqdm.pandas(desc="Parsing History")
        behaviors_df['parsed_history'] = behaviors_df['history'].progress_apply(parse_history)
        print("Finished parsing.")

        behaviors_subset_df = behaviors_df[['user_id', 'time', 'parsed_history', 'parsed_impressions']].copy()
        print(f"\nSelected columns for exploding. Rows: {len(behaviors_subset_df)}")

        print("\nExploding DataFrame based on parsed impressions (can take time and memory)...")
        behaviors_exploded_df = behaviors_subset_df.explode('parsed_impressions', ignore_index=True)
        print(f"DataFrame exploded. New rows: {len(behaviors_exploded_df)}")

        print("Extracting news_id and click from exploded column...")
        behaviors_exploded_df.loc[:, 'news_id'] = behaviors_exploded_df['parsed_impressions'].apply(lambda x: x['news_id'] if isinstance(x, dict) else None)
        behaviors_exploded_df.loc[:, 'click'] = behaviors_exploded_df['parsed_impressions'].apply(lambda x: x['click'] if isinstance(x, dict) else None)
        behaviors_exploded_df = behaviors_exploded_df.drop(columns=['parsed_impressions'])
        behaviors_exploded_df = behaviors_exploded_df.dropna(subset=['news_id'])
        behaviors_exploded_df['click'] = behaviors_exploded_df['click'].astype(int)
        print(f"Columns extracted. Rows after potential dropna: {len(behaviors_exploded_df)}")

        print("\nMerging exploded behaviors with news data...")
        news_subset_df = news_df[['news_id', 'category', 'subcategory', 'title', 'abstract']].copy()
        merged_df = pd.merge(
            behaviors_exploded_df, news_subset_df, on='news_id', how='left'
        )
        print(f"Merge complete. Final DataFrame rows: {len(merged_df)}")

        # --- Add state representation ---
        final_df = create_state_representation(merged_df, news_df, k=1) # Using k=1 for now

        # --- Display final data with state ---
        print("\n--- Final DataFrame with State Column ---")
        # Select key columns to display
        print(final_df[['user_id', 'parsed_history', 'state', 'news_id', 'category', 'click']].head(10))

        print("\n--- State Value Counts (Top 20) ---")
        print(final_df['state'].value_counts().head(20))

        print("\n--- Final DataFrame Info ---")
        final_df.info()

    else:
        print("Data loading failed.")

Starting data loading process...
Loading news data from: data/MINDsmall_train/news.tsv
Loaded 51282 news articles.
Loading behaviors data from: data/MINDsmall_train/behaviors.tsv
Loaded 156965 behavior logs (impressions).

Parsing impressions column...


Parsing Impressions: 100%|██████████| 156965/156965 [00:01<00:00, 114001.07it/s]



Parsing history column...


Parsing History: 100%|██████████████| 156965/156965 [00:00<00:00, 323723.91it/s]


Finished parsing.

Selected columns for exploding. Rows: 156965

Exploding DataFrame based on parsed impressions (can take time and memory)...
DataFrame exploded. New rows: 5843444
Extracting news_id and click from exploded column...
Columns extracted. Rows after potential dropna: 5843444

Merging exploded behaviors with news data...
Merge complete. Final DataFrame rows: 5843444

Creating state representation using last 1 history item(s)...
Creating news_id -> category lookup map...
Lookup map created.
Applying state creation function...


Creating State: 100%|████████████| 5843444/5843444 [00:02<00:00, 2873711.07it/s]


State column created.

--- Final DataFrame with State Column ---
  user_id                                     parsed_history      state  \
0  U13740  [N55189, N42782, N34694, N45794, N18445, N6330...    (news,)   
1  U13740  [N55189, N42782, N34694, N45794, N18445, N6330...    (news,)   
2  U91836  [N31739, N6072, N63045, N23979, N35656, N43353...  (sports,)   
3  U91836  [N31739, N6072, N63045, N23979, N35656, N43353...  (sports,)   
4  U91836  [N31739, N6072, N63045, N23979, N35656, N43353...  (sports,)   
5  U91836  [N31739, N6072, N63045, N23979, N35656, N43353...  (sports,)   
6  U91836  [N31739, N6072, N63045, N23979, N35656, N43353...  (sports,)   
7  U91836  [N31739, N6072, N63045, N23979, N35656, N43353...  (sports,)   
8  U91836  [N31739, N6072, N63045, N23979, N35656, N43353...  (sports,)   
9  U91836  [N31739, N6072, N63045, N23979, N35656, N43353...  (sports,)   

  news_id       category  click  
0  N55689         sports      1  
1  N35729           news      0  
2  N206

In [12]:
# Add this after creating final_df in your main block
print("\nSorting DataFrame by user and time...")
# Ensure 'time' is datetime type for correct sorting
final_df['time'] = pd.to_datetime(final_df['time'], errors='coerce') 
final_df = final_df.sort_values(by=['user_id', 'time']).reset_index(drop=True)
print("DataFrame sorted.")

print("\n--- Sorted DataFrame Head ---")
print(final_df[['user_id', 'time', 'state', 'news_id', 'click']].head(100))


Sorting DataFrame by user and time...
DataFrame sorted.

--- Sorted DataFrame Head ---
   user_id                time  state news_id  click
0     U100 2019-11-12 07:34:12  (tv,)  N61235      0
1     U100 2019-11-12 07:34:12  (tv,)  N54489      0
2     U100 2019-11-12 07:34:12  (tv,)  N42597      0
3     U100 2019-11-12 07:34:12  (tv,)   N7800      1
4     U100 2019-11-12 07:34:12  (tv,)  N61408      0
..     ...                 ...    ...     ...    ...
95    U100 2019-11-12 07:34:12  (tv,)     N21      0
96    U100 2019-11-12 07:34:12  (tv,)  N29945      0
97    U100 2019-11-12 07:34:12  (tv,)  N62360      0
98    U100 2019-11-12 07:34:12  (tv,)  N55606      0
99    U100 2019-11-12 07:34:12  (tv,)  N20486      0

[100 rows x 5 columns]


In [14]:
# q_learning.py

import random
from collections import defaultdict

class QLearningAgent:
    """
    A simple tabular Q-Learning agent.
    """
    def __init__(self, learning_rate=0.1, discount_factor=0.9, epsilon=0.1):
        """
        Initializes the Q-Learning agent.

        Args:
            learning_rate (float): Alpha, the learning rate.
            discount_factor (float): Gamma, the discount factor for future rewards.
            epsilon (float): Exploration rate for epsilon-greedy policy (less relevant for
                             pure offline training, but included for completeness).
        """
        self.alpha = learning_rate
        self.gamma = discount_factor
        self.epsilon = epsilon
        
        # Q-table: nested defaultdict Q[state][action] = value
        # Stores Q-values. Defaults to 0.0 for unseen state-action pairs.
        self.q_table = defaultdict(lambda: defaultdict(float))
        
        # Optional: Keep track of unique states and actions seen
        self.states = set()
        self.actions = set()

    def get_q_value(self, state, action):
        """Gets the Q-value for a state-action pair."""
        return self.q_table[state][action] # Returns 0.0 if not seen

    def get_max_q_value(self, state):
        """Gets the maximum Q-value for a given state over all known actions."""
        # Find the max Q-value only among actions already seen for this state
        action_values = self.q_table.get(state, None)
        if action_values:
            return max(action_values.values())
        else:
            # If the state has not been seen, or no actions have values yet for it
            return 0.0

    def update(self, state, action, reward, next_state):
        """
        Updates the Q-value for the state-action pair using the Q-learning rule.

        Args:
            state: The current state representation.
            action: The action taken (e.g., news_id).
            reward (float): The reward received (e.g., 1 for click, 0 for no click).
            next_state: The next state representation.
        """
        # Add state and action to tracker (optional)
        self.states.add(state)
        self.actions.add(action)
        
        # Q-learning update rule:
        # Q(s, a) = Q(s, a) + alpha * [r + gamma * max_a'(Q(s', a')) - Q(s, a)]
        
        # Current Q-value estimate
        old_value = self.get_q_value(state, action)
        
        # Best Q-value estimate for the next state
        next_max_q = self.get_max_q_value(next_state)
        
        # Calculate the TD target
        target = reward + self.gamma * next_max_q
        
        # Calculate the TD error
        td_error = target - old_value
        
        # Update the Q-value
        new_value = old_value + self.alpha * td_error
        self.q_table[state][action] = new_value

    def choose_action(self, state, available_actions):
        """
        Chooses an action using an epsilon-greedy policy based on current Q-values.
        NOTE: This is mainly for online interaction/evaluation. For offline training,
              we use the actions from the log. For offline evaluation, we might use this.

        Args:
            state: The current state.
            available_actions (list): A list of actions possible in this state (e.g., candidate news).

        Returns:
            The chosen action.
        """
        if not available_actions:
            return None # No actions available

        if random.random() < self.epsilon:
            # Explore: choose a random action
            return random.choice(available_actions)
        else:
            # Exploit: choose the action with the highest Q-value
            q_values = {action: self.get_q_value(state, action) for action in available_actions}
            max_q = max(q_values.values())
            # Choose one of the actions that achieve the max Q-value (handles ties)
            best_actions = [action for action, q in q_values.items() if q == max_q]
            return random.choice(best_actions)

In [16]:
# trainer.py (Conceptual - Needs Refinement)
# Assume data_preprocessing functions and final_df are loaded/created here
# Or load the processed final_df from a file

# --- Load or create final_df (sorted by user_id, time) ---
# final_df = ... (Result from previous steps, ensure it's sorted)

# --- Initialize Agent ---
agent = QLearningAgent(learning_rate=0.1, discount_factor=0.9, epsilon=0.1) 

# --- Training Loop ---
print("Starting training loop...")
num_rows = len(final_df)
for i in tqdm(range(num_rows - 1), desc="Training Q-Learning"): # Iterate up to second-to-last row
    current_row = final_df.iloc[i]
    next_row = final_df.iloc[i+1]

    # Check if the next row belongs to the same user
    if current_row['user_id'] == next_row['user_id']:
        # Get state, action, reward from the current interaction
        state = current_row['state']
        action = current_row['news_id'] # Action is the news article shown
        reward = current_row['click'] # Reward is 1 if clicked, 0 otherwise
        
        # Get the next state from the *next* interaction log for the same user
        next_state = next_row['state']
        
        # Update the agent's Q-table
        agent.update(state, action, reward, next_state)
    else:
        # End of a user's session or transition to a new user
        # Handle the last interaction of the user: next_state is terminal (value 0)
        state = current_row['state']
        action = current_row['news_id']
        reward = current_row['click']
        next_state = None # Or a special terminal state representation if needed
        
        # Update for the terminal transition (next_max_q will be 0)
        agent.update(state, action, reward, next_state) # Pass None or handle in update

print("Training finished.")
# --- Save the Q-table ---
# import pickle
# with open('q_table.pkl', 'wb') as f:
#     pickle.dump(dict(agent.q_table), f) # Convert defaultdict for saving if needed

# print(f"Q-table has {len(agent.q_table)} states.")
# num_q_entries = sum(len(actions) for actions in agent.q_table.values())
# print(f"Q-table has {num_q_entries} state-action entries.")

Starting training loop...


Training Q-Learning: 100%|██████████| 5843443/5843443 [10:32<00:00, 9239.35it/s]

Training finished.
